In [ ]:
!git clone https://github.com/piotrszczypior/backdoor-resnet.git && cd backdoor-resnet && git checkout tsne

In [ ]:
import sys
import os

notebook_dir = os.path.abspath(".")
project_path = os.path.join(notebook_dir, "backdoor-resnet")
sys.path.append(project_path)

In [ ]:
from google.colab import drive
import shutil


def download_weights(
    drive_path="/content/drive/MyDrive/backdoor-resnet/", local_path="weights"
):
    drive.mount("/content/drive")

    os.makedirs(local_path, exist_ok=True)

    for entry in os.listdir(drive_path):
        src = os.path.join(drive_path, entry)

        if os.path.isdir(src):
            continue

        dst = os.path.join(local_path, entry)
        shutil.copy(src, dst)


download_weights()

In [ ]:
import torch
import matplotlib
import os


matplotlib.use("TkAgg")

from src.model import get_resnet_model
from src.dataset import BackdooredDataset
from src.backdoor import gaussian_noise_static_trigger
from src.plot import plt_tsne
from src.utils import extract_features, subsample
import src.loader as loader

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CLASSES = [
    "plane",
    "car",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]


def get_model():
    model = get_resnet_model(10)
    checkpoint = torch.load(
        "weights/weights-tf-cifar100-bd-gauss-static-on-cifar10-clean.pth",
        map_location=DEVICE,
    )
    model.to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])

    model.to(DEVICE)

    return model


def get_backdoored_data_loader():
    test_dataset = BackdooredDataset(
        dataset="CIFAR10",
        train=False,
        transform=loader.get_test_transform_cifar10(),
        backdoor=True,
        trigger_fn=gaussian_noise_static_trigger,
        mode="replace",
        label_mode="clean_label",
        p=1,
    )

    return loader.to_dataloader(test_dataset)

In [ ]:
model = get_model()

clean_data_loader = loader.get_clean_cifar10_test_data_loader()
clean_features, clean_targets = extract_features(model, clean_data_loader)
clean_features, clean_targets = subsample(
    clean_features, clean_targets, target_size=2000
)

backdoor_data_loader = get_backdoored_data_loader()
backdoor_features, backdoor_targets = extract_features(model, backdoor_data_loader)
backdoor_features = subsample(backdoor_features, target_size=2000)

print("control")
plt_tsne(clean_features, clean_targets, backdoor_features)